In [ ]:
import kagglehub
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline
# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# cheek the missing values

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "status")

In [ ]:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# sigmoid in NumPy
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

  # BCE in NumPy
def binary_cross_entropy(y, y_hat):
  epsilon = 1e-15  # Very small number to prevent log(0)
  y_hat = np.clip(y_hat, epsilon, 1 - epsilon) # np.clip(value, min, max)

  loss = -1/len(y) * np.sum(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
  return loss

def gradient_descent(X, y, learning_rate, n_iters=500):
    m, n = X.shape # m rows, n columns (dimensions)
    theta = np.zeros(n) # initialize a zeros weight vector with n dimensions
    losses = []

    for _ in tqdm(range(n_iters), desc="Training Logistic Regression"):
      z = np.dot(X, theta)
      y_hat = sigmoid(z)
      gradient = np.dot(X.T, (y_hat - y)) / m
      theta -= learning_rate * gradient

      loss = binary_cross_entropy(y, y_hat)
      losses.append(loss)

    return theta, losses

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for logistic regression results for each fold
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    theta, losses = gradient_descent(X_train, y_train, learning_rate=0.5, n_iters=500)

    # Validate
    y_pred_proba = sigmoid(np.dot(X_test, theta))
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_losses.append(losses)
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

In [ ]:

# Import models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

sklearn_models = {
  "K-Nearest Neighbors": KNeighborsClassifier(
      n_neighbors=3,  # Number of neighbors to consider
  ),
  "Support Vector Machine": SVC(
      kernel='rbf',  # 'linear', 'poly', 'rbf', 'sigmoid'
      C=0.75  # Regularization parameter
  ),
  "Decision Tree": DecisionTreeClassifier(
      max_depth=3  # Maximum depth of tree (prevents overfitting)
  ),
  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  ),
  "XGBoost": XGBClassifier(
      verbosity=0,
      n_estimators=300,  # Number of boosting rounds
      max_depth=5,
      learning_rate=0.05 # Step size shrinkage
  ),
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

In [ ]:
all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

In [ ]:
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
#use all LOGISTIC REGRESSION Performance :)

print("LOGISTIC REGRESSION Performance:")
print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  Precision: {np.mean(lr_precision):.4f}")
print(f"  Recall:    {np.mean(lr_recall):.4f}")
print(f"  F1-Score:  {np.mean(lr_f1):.4f}")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: